# YOLOv8 鹿種偵測 — Colab (免費 GPU)
在 Colab 執行前，先按 **執行階段 → 變更執行階段類型 → GPU**。
流程：安裝 → 下載資料集 → 訓練 → 評估 → 推論視覺化 → 下載權重。

## 1. 安裝與確認 GPU

In [ ]:
!pip -q install ultralytics roboflow
import torch; print('CUDA available:', torch.cuda.is_available())
!nvidia-smi -L

## 2. 下載鹿類偵測資料集 (Roboflow)
免費註冊 roboflow.com 取得 API key。到 universe.roboflow.com 搜尋 `deer` 選一個偵測資料集，
把 workspace / project / version 換成你選的那個。
> 註：多數公開 deer 資料集是單類別；要真正的鹿『種』需 species 標註的資料集或自行標註。

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key='貼上你的_API_KEY')
ds = rf.workspace('university-itgbn').project('deer-ipsw2').version(2).download('yolov8')
print('data.yaml =>', ds.location + '/data.yaml')
!cat {ds.location}/data.yaml

## 3. 訓練
用 yolov8n 當基底(輕量快)。資料少時 epochs 可設 80~150，並靠內建增強。

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data=ds.location + '/data.yaml', epochs=100, imgsz=640, batch=16,
            patience=20, seed=42, name='deer_yolov8n',
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=5, translate=0.1,
            scale=0.5, fliplr=0.5, mosaic=1.0, mixup=0.1)

## 4. 評估 (mAP)

In [ ]:
metrics = model.val()
print('mAP50-95:', metrics.box.map, '| mAP50:', metrics.box.map50)
from IPython.display import Image, display
display(Image('runs/detect/deer_yolov8n/results.png'))              # 訓練曲線
display(Image('runs/detect/deer_yolov8n/confusion_matrix.png'))     # 混淆矩陣

## 5. 推論與視覺化

In [ ]:
import glob
val_imgs = glob.glob(ds.location + '/valid/images/*.jpg')[:6]
res = model.predict(val_imgs, conf=0.25, save=True)
for p in glob.glob('runs/detect/predict*/*.jpg')[:6]:
    display(Image(p))

## 6. 下載訓練好的權重

In [ ]:
from google.colab import files
files.download('runs/detect/deer_yolov8n/weights/best.pt')